# Lab 20-02 · Designing and Verifying a Two-Stage Op-Amp

**Predict first**, then run. The handout is [README.md](README.md). All numbers use the chapter's illustrative T130 process and Example 20-02.3's specification.

In [1]:
import math
import re
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import opamp as o
import gmid as g

spec, ch = o.Spec(), o.Choices()
d = o.design_two_stage(spec, ch)
for k in ("gm1", "iss", "sr", "gm6", "i6", "c1", "a0_db", "swing", "noise", "itot", "fom_s"):
    print(f"{k:6s} = {d[k]:.4g}")


def ngspice(netlist):
    """Run ngspice in batch mode and return the 'name = value' measurements it prints."""
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

gm1    = 0.0001885
iss    = 2.094e-05
sr     = 3.491e+07
gm6    = 0.001558
i6     = 0.0001558
c1     = 1.107e-13
a0_db  = 63.17
swing  = 0.8598
noise  = 1.353e-08
itot   = 0.0001768
fom_s  = 565.8


## Part 1 · Where the current goes

*Predict* whether the total current rises or falls when M6 runs at 6 S/A instead of 10 S/A, and where the slew-constrained minimum lies.

In [2]:
ccs = np.linspace(0.35e-12, 1.6e-12, 200)
for gi, ls in ((10.0, "-"), (6.0, "--")):
    c = o.Choices(gmid6=gi)
    plt.plot(ccs * 1e12, [o.total_current(x, spec, c) * 1e6 for x in ccs], ls, label=f"(g_m/I_D)_6 = {gi:g} S/A")
    copt = o.optimum_cc(spec, c)
    print(f"(g_m/I_D)_6 = {gi:4.1f} S/A: optimum C_c = {copt * 1e12:.3f} pF, total {o.total_current(copt, spec, c) * 1e6:.1f} uA")
plt.xlabel("C_c (pF)"); plt.ylabel("I_SS + I_6 (uA)"); plt.legend(); plt.show()

wu = 2 * math.pi * spec.gbw


def best_total(gi):
    """Minimum I_SS + I_6 over C_c with the output-slew condition I_6 >= I_SS (1 + C_L/C_c)."""
    c = o.Choices(gmid6=gi)
    ic6 = g.ic_for_gm_over_id(gi, c.nmos, c.l6)
    wt6 = 2 * math.pi * g.ft(ic6, c.nmos, c.l6)
    best = None
    for cc in np.arange(0.4e-12, 2.0e-12, 0.002e-12):
        try:
            gm6 = o.gm6_for_p2(2 * wu, cc, spec.cl, c.cp, wt6)
        except ValueError:
            continue
        iss, i6 = 2 * wu * cc / 18, gm6 / gi
        if i6 >= iss * (1 + spec.cl / cc) and (best is None or iss + i6 < best[0]):
            best = (iss + i6, cc)
    return best


gis = np.arange(6, 26.01, 1.0)
res = [best_total(gi) for gi in gis]
for gi, r in zip(gis, res):
    print(f"(g_m/I_D)_6 = {gi:4.1f} S/A: {r[0] * 1e6:6.1f} uA at C_c = {r[1] * 1e12:.3f} pF")
i = int(np.argmin([r[0] for r in res]))
print(f"minimum near {gis[i]:.0f} S/A: {res[i][0] * 1e6:.0f} uA ({(1 - res[i][0] / d['itot']) * 100:.0f} % below Example 20-02.3)")

(g_m/I_D)_6 = 10.0 S/A: optimum C_c = 0.672 pF, total 176.4 uA


(g_m/I_D)_6 =  6.0 S/A: optimum C_c = 0.672 pF, total 262.0 uA
(g_m/I_D)_6 =  6.0 S/A:  262.0 uA at C_c = 0.672 pF
(g_m/I_D)_6 =  7.0 S/A:  230.8 uA at C_c = 0.662 pF
(g_m/I_D)_6 =  8.0 S/A:  207.7 uA at C_c = 0.660 pF
(g_m/I_D)_6 =  9.0 S/A:  190.1 uA at C_c = 0.664 pF
(g_m/I_D)_6 = 10.0 S/A:  176.4 uA at C_c = 0.672 pF
(g_m/I_D)_6 = 11.0 S/A:  165.6 uA at C_c = 0.686 pF
(g_m/I_D)_6 = 12.0 S/A:  157.0 uA at C_c = 0.704 pF
(g_m/I_D)_6 = 13.0 S/A:  150.2 uA at C_c = 0.726 pF
(g_m/I_D)_6 = 14.0 S/A:  144.9 uA at C_c = 0.752 pF
(g_m/I_D)_6 = 15.0 S/A:  140.9 uA at C_c = 0.786 pF
(g_m/I_D)_6 = 16.0 S/A:  138.1 uA at C_c = 0.826 pF
(g_m/I_D)_6 = 17.0 S/A:  136.5 uA at C_c = 0.874 pF
(g_m/I_D)_6 = 18.0 S/A:  136.0 uA at C_c = 0.932 pF
(g_m/I_D)_6 = 19.0 S/A:  136.8 uA at C_c = 0.960 pF
(g_m/I_D)_6 = 20.0 S/A:  139.5 uA at C_c = 0.996 pF
(g_m/I_D)_6 = 21.0 S/A:  144.1 uA at C_c = 1.062 pF
(g_m/I_D)_6 = 22.0 S/A:  151.4 uA at C_c = 1.166 pF
(g_m/I_D)_6 = 23.0 S/A:  162.4 uA at C_c = 1.326 pF
(

## Part 2 · Exact margins in ngspice

*Predict* the crossover and phase margin for R_z = 0, 1/g_m6 and the cancelling value (Example 20-02.4).

In [3]:
for rz, name in ((0.0, "a"), (d["rz"], "b"), (d["rz_cancel"], "c")):
    num, den = o.miller_tf(d["gm1"], d["gm6"], d["r1"], d["r2"], d["c1"], spec.cl, ch.cc, rz)
    m = o.margins(o.poly_tf(num, den))
    print(f"({name}) R_z = {rz:7.1f} ohm: crossover {m['fc'] / 1e6:6.2f} MHz, PM {m['pm']:5.2f} deg")
sim = ngspice("miller.cir")
print({k: round(v, 3) for k, v in sim.items() if k.startswith(("fc", "pm", "a0"))})

(a) R_z =     0.0 ohm: crossover  43.64 MHz, PM 61.75 deg


(b) R_z =   641.8 ohm: crossover  43.62 MHz, PM 67.59 deg


(c) R_z =  2781.1 ohm: crossover  47.03 MHz, PM 85.98 deg
{'a0': 1440.004, 'fca': 43642460.0, 'fcb': 43619100.0, 'fcc': 47025390.0, 'pma': 61.754, 'pmb': 67.592, 'pmc': 85.979}


## Part 3 · Slew rate

*Predict* the slew rate from I_SS/C_c, then from the steady-ramp formula (I_SS/C_c)/(1 + 1/A_2).

In [4]:
print(f"I_SS/C_c = {d['sr'] / 1e6:.2f} V/us; steady-ramp estimate {d['sr'] / (1 + 1 / d['a2']) / 1e6:.2f} V/us")
sim = ngspice("slew.cir")
print(f"ngspice: rising {sim['srr'] / 1e6:.2f} V/us, falling {sim['srf'] / 1e6:.2f} V/us")

I_SS/C_c = 34.91 V/us; steady-ramp estimate 33.24 V/us


ngspice: rising 32.72 V/us, falling 32.72 V/us


## Part 4 · Supply rejection

*Predict* PSRR- at 1 kHz and 1 MHz from section 3.10 (PSRR(0) = g_m1 R_1 g_m6 r_o6, falling as f_u/f above about 17 kHz).

In [5]:
gm1, gm6, r1, c1, cc, cl = d["gm1"], d["gm6"], d["r1"], d["c1"], ch.cc, spec.cl
ro6 = 4 / d["i6"]
f = np.logspace(1, 8, 300)
sv = 2j * np.pi * f
inv = (1 / r1 / ro6 + sv * (cc * gm6 + c1 / ro6 + cc * (1 / r1 + 1 / ro6)) + sv**2 * c1 * cc) / (gm1 * (gm6 - sv * cc))
plt.semilogx(f, -20 * np.log10(np.abs(inv)), label="PSRR- (section 3.10)")
plt.semilogx(f, 20 * np.log10(spec.gbw / f), "--", label="f_u / f")
plt.ylim(0, 80); plt.xlabel("frequency (Hz)"); plt.ylabel("PSRR- (dB)"); plt.legend(); plt.show()
sim = ngspice("psrr.cir")
print(f"ngspice follower: {sim['rej1k']:.2f} dB at 1 kHz, {sim['rej1m']:.2f} dB at 1 MHz")

ngspice follower: 69.18 dB at 1 kHz, 33.72 dB at 1 MHz


## Part 5 · The pole pair

*Predict* which of Example 20-02.5's two designs rings in a step response.

In [6]:
gm6i = 1e-3
c1i = 30e-15 + gm6i / (2 * math.pi * d["ft6"])
t = np.linspace(0, 100e-9, 20001)
for gmc in (d["gm1"], 600e-6):
    h = o.indirect_tf(d["gm1"], gm6i, gmc, d["r1"], 20e3, 1e6, c1i, cl, cc, 20e-15)
    m = o.margins(h)
    print(f"g_mc = {gmc * 1e6:6.1f} uS: crossover {m['fc'] / 1e6:.2f} MHz, PM {m['pm']:.1f} deg, GM {m['gm_db']:.2f} dB")
sim = ngspice("indirect.cir")
print({k: round(v, 3) for k, v in sim.items() if k.startswith(("pm", "gm"))})


def closed_loop_step(gmc):
    """Closed-loop step response of the indirect design from its exact transfer function (sympy)."""
    import sympy as sp
    s = sp.symbols("s")
    y = sp.Matrix([[1 / d["r1"] + s * c1i, -gmc, 0], [0, gmc + 1e-6 + s * (20e-15 + cc), -s * cc],
                   [gm6i, -s * cc, 1 / 20e3 + s * (cl + cc)]])
    hs = sp.together(y.LUsolve(sp.Matrix([0, -d["gm1"], 0]))[2])
    nn, dd = sp.fraction(hs)
    nc = np.array([float(x) for x in sp.Poly(sp.expand(nn), s).all_coeffs()])
    dc = np.array([float(x) for x in sp.Poly(sp.expand(dd), s).all_coeffs()])
    _, yv = signal.step((nc, np.polyadd(dc, nc)), T=t)
    return yv / (nc[-1] / (dc[-1] + nc[-1]))


for gmc in (d["gm1"], 600e-6):
    yv = closed_loop_step(gmc)
    plt.plot(t * 1e9, yv, label=f"g_mc = {gmc * 1e6:.0f} uS, overshoot {max(0, yv.max() - 1) * 100:.1f} %")
plt.xlabel("time (ns)"); plt.ylabel("normalized output"); plt.legend(); plt.show()

g_mc =  188.5 uS: crossover 56.31 MHz, PM 80.8 deg, GM 1.03 dB


g_mc =  600.0 uS: crossover 50.59 MHz, PM 83.2 deg, GM 10.28 dB
{'pmb': 80.823, 'gmb': 1.034, 'pmc': 83.193, 'gmc': 10.277}


## Part 6 · Re-tuning nested Miller

*Predict* how much more output-stage current than the Butterworth value it takes to reach PM >= 60 deg and GM >= 10 dB with Example 20-02.6's parasitics.

In [7]:
gm3b = 4 * wu * cl
found = None
for f3 in np.round(np.arange(1.0, 3.0001, 0.1), 2):
    gm3 = f3 * gm3b
    cgg3 = gm3 / (2 * math.pi * d["ft6"])
    for g2f in (1, 1.5, 2, 3):
        for cm2 in np.round(np.arange(0.1, 0.8001, 0.05), 2):
            h = o.nmc_tf(d["gm1"], g2f * d["gm1"], gm3, 25 / d["gm1"], 25 / (g2f * d["gm1"]), 20 / gm3,
                         30e-15, 30e-15 + cgg3, cl, 0.6e-12, cm2 * 1e-12)
            m = o.margins(h, points=8001)
            if m["pm"] >= 60 and m["gm_db"] is not None and m["gm_db"] >= 10 and found is None:
                found = (f3, g2f, cm2, m)
    if found:
        break
f3, g2f, cm2, m = found
print(f"g_m3 = {f3} x Butterworth ({f3 * gm3b / 10 * 1e6:.1f} uA at 10 S/A), g_m2 = {g2f} g_m1, C_m2 = {cm2} pF: "
      f"PM {m['pm']:.1f} deg, GM {m['gm_db']:.1f} dB")
sim = ngspice("nmc.cir")
print({k: round(v, 2) for k, v in sim.items() if k.startswith(("pm", "gm"))})

g_m3 = 1.8 x Butterworth (452.4 uA at 10 S/A), g_m2 = 2 g_m1, C_m2 = 0.5 pF: PM 61.8 deg, GM 10.1 dB
{'pmb': 58.16, 'gmb': 9.2, 'pmc': 49.35, 'gmc': 5.73, 'pmd': 66.65, 'gmd': 10.37}
